# 07 - BAF Cross-Dataset Rule and Explanation Replication

In [1]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git"
KAGGLE_PROJECT_DIR = Path("/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection")

if KAGGLE:
    os.environ.setdefault("THESIS_QUICK_RUN", "0")
    os.environ.setdefault("THESIS_SYNTHETIC_FALLBACK", "0")

def sync_kaggle_project() -> Path:
    """Use one current working clone; never import source bundled in an input artifact."""
    if not KAGGLE_PROJECT_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(KAGGLE_PROJECT_DIR)],
            check=True,
        )
    else:
        if not (KAGGLE_PROJECT_DIR / ".git").is_dir():
            raise RuntimeError(
                f"Kaggle project path exists but is not a Git clone: {KAGGLE_PROJECT_DIR}"
            )
        subprocess.run(
            ["git", "-C", str(KAGGLE_PROJECT_DIR), "pull", "--ff-only", "origin", "main"],
            check=True,
        )
    return KAGGLE_PROJECT_DIR.resolve()

def find_project_root() -> Path | None:
    direct_candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in direct_candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate.resolve()
    return None

PROJECT_ROOT = sync_kaggle_project() if KAGGLE else find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

project_root_string = str(PROJECT_ROOT)
while project_root_string in sys.path:
    sys.path.remove(project_root_string)
sys.path.insert(0, project_root_string)

# Run All can reuse a live Kaggle kernel. Remove previously imported project
# modules so an updated working clone cannot be shadowed by stale objects.
for module_name in tuple(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

AUDIT_SOURCE_FILES = (
    "scripts/generate_notebooks.py",
    "src/artifacts.py",
    "src/data/dataset.py",
    "src/data/preprocessing.py",
    "src/experiment.py",
    "src/explanation/explanation_metrics.py",
    "src/explanation/rule_explainer.py",
    "src/logic/fraud_rules.py",
    "src/logic/knowledge_base.py",
    "src/logic/predicates.py",
    "src/logic/tensor_logic.py",
)

def audit_pipeline_fingerprint(config_path: Path) -> str:
    paths = [PROJECT_ROOT / relative for relative in AUDIT_SOURCE_FILES]
    paths.append(Path(config_path))
    missing = [str(path) for path in paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Files required for the audit-pipeline fingerprint are missing: {missing}")
    digest = hashlib.sha256()
    for path in sorted(paths, key=lambda item: item.relative_to(PROJECT_ROOT).as_posix()):
        relative = path.relative_to(PROJECT_ROOT).as_posix()
        digest.update(relative.encode("utf-8"))
        digest.update(b"\0")
        digest.update(path.read_bytes())
        digest.update(b"\0")
    return digest.hexdigest()

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"
INPUT_ROOTS = [OUTPUT_BASE, PROJECT_ROOT / "results/runs/notebooks"]
if Path("/kaggle/input").exists():
    INPUT_ROOTS.append(Path("/kaggle/input"))

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_COMMIT = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print({
    "project_root": str(PROJECT_ROOT),
    "project_source_policy": "working_clone_main" if KAGGLE else "local_project_root",
    "git_commit": GIT_COMMIT,
    "quick_run": QUICK_RUN,
    "synthetic_fallback": ALLOW_SYNTHETIC_FALLBACK,
    "kaggle": KAGGLE,
})

Cloning into '/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection'...


{'project_root': '/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection', 'project_source_policy': 'working_clone_main', 'git_commit': '1c833b23e6ce4f00c49513e79ad7f7715615986a', 'quick_run': False, 'synthetic_fallback': False, 'kaggle': True}


## Thiết lập

Notebook chỉ đọc frozen BAF reference predictor từ Notebook 03. Rules fit trên train months 0-4,
được kiểm tra trên validation month 5 và locked test months 6-7.
Đây là replication/portability evaluation với BAF-specific predictor và rule base, không phải
việc chuyển nguyên model hoặc rules từ IEEE-CIS. BAF là privacy-preserving synthetic benchmark.

In [2]:
preflight_manifests = []
preflight_artifacts = []
for root_value in INPUT_ROOTS:
    root = Path(root_value)
    if not root.exists():
        continue
    manifest_paths = [root] if root.is_file() and root.name == "frozen_reference_manifest.json" else list(root.glob("**/frozen_reference_manifest.json"))
    for manifest_path in manifest_paths:
        try:
            manifest_payload = json.loads(manifest_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if str(manifest_payload.get("dataset_name", "")).lower() != "baf".lower():
            continue
        preflight_manifests.append(manifest_path.resolve())
        artifact_path = manifest_path.parent / str(manifest_payload.get("artifact_file", ""))
        if artifact_path.exists():
            preflight_artifacts.append(artifact_path.resolve())

preflight_table = pd.DataFrame({
    "manifest": [str(path) for path in sorted(set(preflight_manifests))],
})
display(preflight_table)
print("Frozen artifacts:")
for path in sorted(set(preflight_artifacts)):
    print(path)
if not preflight_manifests or not preflight_artifacts:
    raise FileNotFoundError(
        "No complete baf frozen artifact was found below INPUT_ROOTS. "
        "On Kaggle, attach the corresponding benchmark notebook output; locally, place it below results/runs/notebooks."
    )

,manifest
0,/kaggle/input/notebooks/giahuytranviet/03-baf-model-benchmarks/thesis_outputs/03_baf_model_benchmarks/frozen_referen...


Frozen artifacts:
/kaggle/input/notebooks/giahuytranviet/03-baf-model-benchmarks/thesis_outputs/03_baf_model_benchmarks/frozen_reference_artifact.npz


In [3]:
from src.artifacts import assert_frozen_alignment, load_frozen_reference_artifact, sha256_file
from src.data import load_config, prepare_dataset
from src.experiment import load_experiment_data

config = load_config(PROJECT_ROOT / "configs/baf.yaml")
frame, data_source = load_experiment_data(
    config, max_rows=12000 if QUICK_RUN else None,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
    synthetic_rows=12000 if QUICK_RUN else 6000,
)
prepared = prepare_dataset(frame, config)
artifact = load_frozen_reference_artifact(
    "baf", expected_config=config,
    search_roots=[OUTPUT_BASE / "03_baf_model_benchmarks", *INPUT_ROOTS],
)
assert_frozen_alignment(artifact, prepared.y_validation, prepared.y_test)
if bool(artifact["manifest"]["quick_run"]) != QUICK_RUN:
    raise ValueError("Notebook mode and frozen artifact quick_run flag do not match")
if int(artifact["manifest"]["reference_seed"]) != int(config["evaluation"]["reference_seed"]):
    raise ValueError("Frozen artifact reference seed does not match the locked dataset protocol")
if not QUICK_RUN and str(artifact["manifest"].get("data_source", "")).lower() == "synthetic":
    raise ValueError("Full thesis evaluation cannot consume a synthetic-fallback frozen artifact")
probabilities = artifact["test_probability"]
threshold = float(artifact["manifest"]["threshold"])
print({
    "data_source": data_source,
    "benchmark_type": frame.attrs.get("benchmark_type", "privacy_preserving_synthetic_benchmark"),
    "reference_model": artifact["manifest"]["model"],
    "reference_seed": artifact["manifest"]["reference_seed"],
    "calibration_method": artifact["manifest"]["calibration_method"],
    "threshold": threshold,
    "artifact": str(artifact["artifact_path"]),
})

def write_upstream_lineage(destination, notebook_id, output_files, config_path):
    output_files = list(output_files)
    lineage = {
        "notebook_id": notebook_id,
        "git_commit": GIT_COMMIT,
        "dataset_name": artifact["manifest"]["dataset_name"],
        "data_source": data_source,
        "frozen_data_source": artifact["manifest"].get("data_source"),
        "quick_run": QUICK_RUN,
        "reference_model_key": artifact["manifest"]["model_key"],
        "reference_seed": artifact["manifest"]["reference_seed"],
        "config_sha256": artifact["manifest"]["config_sha256"],
        "audit_source_sha256": audit_pipeline_fingerprint(config_path),
        "frozen_manifest_sha256": sha256_file(artifact["manifest_path"]),
        "frozen_artifact_sha256": sha256_file(artifact["artifact_path"]),
        "output_files": output_files,
        "output_sha256": {
            name: sha256_file(Path(destination) / name) for name in output_files
        },
    }
    lineage_path = Path(destination) / "upstream_lineage.json"
    lineage_path.write_text(json.dumps(lineage, indent=2), encoding="utf-8")
    return lineage_path

{'data_source': 'real', 'benchmark_type': 'privacy_preserving_synthetic_benchmark', 'reference_model': 'lightgbm', 'reference_seed': 42, 'calibration_method': 'isotonic', 'threshold': 0.07246376811594203, 'artifact': '/kaggle/input/notebooks/giahuytranviet/03-baf-model-benchmarks/thesis_outputs/03_baf_model_benchmarks/frozen_reference_artifact.npz'}


## Rule and explanation results

In [4]:
from src.explanation import (
    RuleExplainer, bootstrap_explanation_precision_gain,
    explanation_quality_metrics, rule_quality_table,
)
from src.logic import FraudKnowledgeBase, FraudRuleEngine

output_dir = OUTPUT_BASE / "07_baf_ltn_generalization"
output_dir.mkdir(parents=True, exist_ok=True)
target = config["dataset"]["target_column"]
engine = FraudRuleEngine(config["logic"]["rules"]).fit(prepared.train_frame, target)
knowledge_base = FraudKnowledgeBase(engine)
fitted_thresholds = engine.fitted_thresholds()
rationale_rows = []
for definition in config["logic"]["rules"]:
    operators = [condition["operator"] for condition in definition["conditions"]]
    rationale_rows.append({
        "rule": definition["name"],
        "description": definition.get("description", ""),
        "knowledge_type": (
            "data-informed fuzzy hypothesis" if "category_risk" in operators
            else "domain hypothesis with train-fitted thresholds"
            if any(operator.endswith("_quantile") for operator in operators)
            else "configured domain hypothesis"
        ),
        "threshold_source": (
            "training-split quantile plus any explicit configured condition"
            if any(operator.endswith("_quantile") for operator in operators)
            else "explicit configured value"
        ),
        "limitation": (
            "BAF-specific association; this rule is not transferred from IEEE-CIS and is not causal."
        ),
    })
rule_rationale = pd.DataFrame(rationale_rows)
validation_truth = engine.evaluate(prepared.validation_frame)
test_truth = engine.evaluate(prepared.test_frame)
activation = float(config["logic"]["activation_threshold"])
validation_quality = rule_quality_table(validation_truth, prepared.y_validation, activation).assign(split="validation")
test_quality = rule_quality_table(test_truth, prepared.y_test, activation).assign(split="test")
rule_quality = pd.concat([validation_quality, test_quality], ignore_index=True)
satisfaction = pd.DataFrame([
    {"split": "train", **knowledge_base.satisfaction_breakdown(prepared.train_frame, target)},
    {"split": "validation", **knowledge_base.satisfaction_breakdown(prepared.validation_frame, target)},
    {"split": "test", **knowledge_base.satisfaction_breakdown(prepared.test_frame, target)},
])
explainer = RuleExplainer(engine, activation, config["logic"]["top_k_rules"])
explanations = explainer.explain(prepared.test_frame, probabilities, threshold)
quality = explanation_quality_metrics(explanations, prepared.y_test, probabilities, threshold)
quality.update(bootstrap_explanation_precision_gain(
    explanations, prepared.y_test, probabilities, threshold,
    n_bootstrap=config["evaluation"]["bootstrap_iterations"], seed=config["project"]["seed"],
))
explanation_quality = pd.DataFrame([quality])
display(
    fitted_thresholds,
    rule_rationale,
    rule_quality.round(4),
    satisfaction.round(4),
    explanation_quality.round(4),
)
rule_quality.to_csv(output_dir / "baf_rule_quality.csv", index=False)
satisfaction.to_csv(output_dir / "baf_knowledge_base_satisfaction.csv", index=False)
explanation_quality.to_csv(output_dir / "baf_explanation_quality.csv", index=False)
fitted_thresholds.to_csv(output_dir / "baf_fitted_rule_thresholds.csv", index=False)
rule_rationale.to_csv(output_dir / "baf_rule_rationale.csv", index=False)

,rule,feature,operator,configured_value,fitted_threshold,softness,softness_mode,fitted_missing_value
0,high_velocity,velocity_6h,greater_quantile,0.95,11914.976769,0.20,relative,6274.609077
1,device_email_linkage,device_distinct_emails_8w,greater_quantile,0.90,1.000000,0.20,relative,1.000000
2,foreign_high_limit_request,foreign_request,equals,1.00,1.000000,0.05,absolute,NaN
3,foreign_high_limit_request,proposed_credit_limit,greater_quantile,0.85,1500.000000,0.20,relative,200.000000
4,short_session_high_risk,session_length_in_minutes,less_quantile,0.10,1.899425,0.20,relative,5.318534
5,short_session_high_risk,credit_risk_score,greater_quantile,0.85,203.000000,0.20,relative,117.000000
6,young_high_income_request,customer_age,less,25.00,25.000000,3.00,absolute,30.000000
7,young_high_income_request,income,greater_quantile,0.90,0.900000,0.20,relative,0.600000


,rule,description,knowledge_type,threshold_source,limitation
0,high_velocity,Recent application velocity is unusually high.,domain hypothesis with train-fitted thresholds,training-split quantile plus any explicit configured condition,BAF-specific association; this rule is not transferred from IEEE-CIS and is not causal.
1,device_email_linkage,The device is linked to an unusually high number of email addresses.,domain hypothesis with train-fitted thresholds,training-split quantile plus any explicit configured condition,BAF-specific association; this rule is not transferred from IEEE-CIS and is not causal.
2,foreign_high_limit_request,A foreign request is associated with a high proposed credit limit.,domain hypothesis with train-fitted thresholds,training-split quantile plus any explicit configured condition,BAF-specific association; this rule is not transferred from IEEE-CIS and is not causal.
3,short_session_high_risk,A short application session has an elevated credit-risk score.,domain hypothesis with train-fitted thresholds,training-split quantile plus any explicit configured condition,BAF-specific association; this rule is not transferred from IEEE-CIS and is not causal.
4,young_high_income_request,A young applicant reports unusually high income.,domain hypothesis with train-fitted thresholds,training-split quantile plus any explicit configured condition,BAF-specific association; this rule is not transferred from IEEE-CIS and is not causal.


,rule,evaluated_rows,coverage,active_count,fraud_precision,lift,mean_truth,rule_auc,split
0,foreign_high_limit_request,119323,0.0003,37,0.1892,15.9990,0.0024,0.5145,validation
1,device_email_linkage,119323,0.0151,1799,0.0650,5.4999,0.5054,0.5341,validation
2,short_session_high_risk,119323,0.0225,2682,0.0246,2.0810,0.0687,0.4979,validation
3,high_velocity,119323,0.0041,486,0.0082,0.6960,0.0721,0.4702,validation
4,young_high_income_request,119323,0.0000,0,NaN,NaN,0.0910,0.3963,validation
5,foreign_high_limit_request,205011,0.0004,74,0.2432,17.3272,0.0024,0.5137,test
6,device_email_linkage,205011,0.0133,2719,0.0585,4.1656,0.5044,0.5195,test
7,short_session_high_risk,205011,0.0199,4070,0.0334,2.3803,0.0636,0.4947,test
8,high_velocity,205011,0.0043,879,0.0114,0.8104,0.0482,0.4729,test
9,young_high_income_request,205011,0.0000,0,NaN,NaN,0.0967,0.3848,test


,split,overall_satisfaction,positive_satisfaction,negative_satisfaction,balanced_satisfaction
0,train,0.4794,0.5629,0.4786,0.5207
1,validation,0.4883,0.5534,0.4875,0.5204
2,test,0.4899,0.5393,0.4892,0.5142


,test_rows,predicted_alert_count,non_alert_count,predicted_alert_fraud_count,explained_count,zero_rule_count,explained_alert_count,explained_alert_fraud_count,unsupported_alert_count,rule_evidence_without_alert_count,explanation_coverage_all,zero_rule_fraction,explanation_coverage_alerts,unsupported_alert_rate,rule_evidence_without_alert_rate,mean_rule_count,mean_active_rule_count,mean_displayed_rule_count,mean_active_rule_fraction,sparsity,rule_sparsity,prediction_rule_consistency,balanced_prediction_rule_consistency,contradiction_rate,explained_alert_precision,all_alert_precision,explained_alert_precision_gain,precision_gain_bootstrap_mean,precision_gain_ci_low,precision_gain_ci_high,bootstrap_valid_iterations
0,205011,7480,197531,1274,7685,197326,861,192,6619,6824,0.0375,0.9625,0.1151,0.8849,0.0345,0.0378,0.0378,0.0378,0.0076,0.9924,0.9924,0.9344,0.5403,0.0656,0.223,0.1703,0.0527,0.0525,0.0276,0.0777,1000.0


In [5]:
stability = validation_quality.merge(test_quality, on="rule", suffixes=("_validation", "_test"))
stability["coverage_delta"] = stability["coverage_test"] - stability["coverage_validation"]
stability["lift_delta"] = stability["lift_test"] - stability["lift_validation"]
display(stability[["rule", "coverage_delta", "lift_delta"]].round(4))
stability.to_csv(output_dir / "baf_rule_stability.csv", index=False)
lineage_path = write_upstream_lineage(
    output_dir,
    "07_BAF_Cross_Dataset_Rule_and_Explanation_Replication",
    [
        "baf_rule_quality.csv", "baf_knowledge_base_satisfaction.csv",
        "baf_explanation_quality.csv", "baf_fitted_rule_thresholds.csv",
        "baf_rule_rationale.csv", "baf_rule_stability.csv",
    ],
    PROJECT_ROOT / "configs/baf.yaml",
)
print({"upstream_lineage": str(lineage_path)})

,rule,coverage_delta,lift_delta
0,foreign_high_limit_request,0.0001,1.3281
1,device_email_linkage,-0.0018,-1.3343
2,short_session_high_risk,-0.0026,0.2992
3,high_velocity,0.0002,0.1144
4,young_high_income_request,0.0000,NaN


{'upstream_lineage': '/kaggle/working/thesis_outputs/07_baf_ltn_generalization/upstream_lineage.json'}


## Takeaways

In [6]:
eligible_rules = test_quality.dropna(subset=["lift"]).query("active_count > 0")
strongest_rule = eligible_rules.sort_values("lift", ascending=False).iloc[0]
display(Markdown(
    f"- Frozen predictor: **{artifact['manifest']['model']}**, seed **{artifact['manifest']['reference_seed']}**.\n"
    f"- Highest observed BAF test lift: **{strongest_rule['rule']} = {strongest_rule['lift']:.3f}** "
    f"with **{int(strongest_rule['active_count'])}** active rows.\n"
    f"- Alert explanation coverage: **{quality['explanation_coverage_alerts']:.3f}**; "
    f"unsupported-alert rate: **{quality['unsupported_alert_rate']:.3f}**.\n"
    "- This is BAF-specific framework replication. It does not establish model/rule transfer, causality, or production generalization."
))

- Frozen predictor: **lightgbm**, seed **42**.
- Highest observed BAF test lift: **foreign_high_limit_request = 17.327** with **74** active rows.
- Alert explanation coverage: **0.115**; unsupported-alert rate: **0.885**.
- This is BAF-specific framework replication. It does not establish model/rule transfer, causality, or production generalization.